# Modules and data import

In [ ]:
# For online projects files edits and no kernel and modules reloaded
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload
# %pip install -U pip

In [ ]:
import pandas as pd

# Load processed data
oil, stores, transactions, sales, X_exog_train, X_exog_test = (
    pd.read_parquet("../data/processed/oil.pq"),
    pd.read_parquet("../data/processed/stores.pq"),
    pd.read_parquet("../data/processed/transactions.pq"),
    pd.read_parquet("../data/processed/sales.pq"),
    pd.read_parquet("../data/processed/sales_exog_train.pq"),
    pd.read_parquet("../data/processed/sales_exog_test.pq"),
)

In [ ]:
from pathlib import Path

import plotly.express as px
import plotly.io as pio
from joblib import cpu_count, dump, load
from sktime.forecasting.base import ForecastingHorizon

import megatron.config as config

# Set global config variables for running session
config.set_config(
    SEASONAL_PERIOD=7,
    MAX_LAG_W_SIZE=30,
    COUNTRY="EC",
    MIN_DATE=oil.index.min(),
    MAX_DATE=oil.index.max(),
    FH_SIZE=X_exog_test.index.get_level_values(-1).nunique(),
)

from megatron.clusterers import (  # noqa: E402
    IntermittentLumpyClusterer,
    SmoothErraticClusterer,
)
from megatron.forecasters import CommonForecaster  # noqa: E402
from megatron.pipelines import E2EForecaster  # noqa: E402
from megatron.transformers import (  # noqa: E402
    ChangePointDetector,
    DemandClassifier,
    ExogenousDataTransformer,
    Mapper,
    OutlierDetector,
    PlateauDetector,
)
from megatron.visualization import (  # noqa: E402
    classifiedSeriesPlot,
    clusteredSeriesPlot,
    forecastedSeriesPlot,
    seriesPlot,
)

fh = ForecastingHorizon(
    values=[*range(1, config.FH_SIZE + 1)],  # type: ignore
    is_relative=True,
    freq="D",
)
px.defaults.width, px.defaults.height = config.FIG_WIDTH, config.FIG_HEIGHT  # type: ignore
pio.renderers.default = "png"

# Exogenous features

## Transactions

In [ ]:
# Since transactions data're available only on train period fitting an intermediate
# forecasting model is required in order to use this data as a complete
# exogenous variable for the main model quality enhancement

# Proposed solution disclosing CommonPipeline stepwise strategy

temp = transactions.copy()
temp.head()

### Exogenous data

In [ ]:
# Exogenous data for both train and test periods construction
X_temp_train = X_exog_train.groupby(["store_nbr", "date"]).sum().join(oil).join(stores)
X_temp_test = X_exog_test.groupby(["store_nbr", "date"]).sum().join(oil).join(stores)

X_temp_train.head()

### Multiindex reduction

In [ ]:
# Replacing multiindex with a single index to avoid further pandas index manipulation
# issues
mapper = Mapper()
temp = mapper.fit_transform(temp)
X_temp_train = mapper.transform(X_temp_train)
X_temp_test = mapper.transform(X_temp_test)

### Classification

In [ ]:
# Series behavior classification to define an appropriate processing and modeling
result = DemandClassifier().fit_transform(temp)
result["class"].value_counts()  # type: ignore

In [ ]:
demand, value = "smooth", "transactions"

### Plateau detection

In [ ]:
# Defining a temporal inactivity expressed as a plateau - zero values sequence with
# a specific length and leave only data which follows the last plateau if any defined
pld = PlateauDetector(w=2 * config.SEASONAL_PERIOD, value=0, truncate=True)  # type: ignore
temp = pld.fit_transform(temp)

### Change point detection

In [ ]:
# Defining an inflection point with specific requirements where the most significant
# trend behavior change was occurred
seriesPlot(
    data=temp,  # type: ignore
    demand=demand,
    n_series=4,
    title=f"Ecuador store {value} with change point detection",
    w=config.MIN_LENGTH,  # type: ignore
    cpd=True,
    seed=555,
)

In [ ]:
# Leave only data which follows right after this point if any exists
cpd = ChangePointDetector(w=config.MIN_LENGTH, truncate=True)  # type: ignore
temp = cpd.fit_transform(temp)

### Outliers detection

In [ ]:
# Defining outliers as a significant temporal demand with holidays and promotions
# data taking into account and remove selected points that are not related to holidays
# and promotion days
od = OutlierDetector(demand=demand, exog_column="on_promotion", truncate=True)
temp = od.fit_transform(temp.join(X_temp_train))  # type: ignore

### Missing values imputation

In [ ]:
# Interpolating missing values if any present
index = temp.index  # type: ignore
temp = temp.groupby(index.names[0]).transform(  # type: ignore
    lambda x: x.interpolate(method="linear").bfill().ffill()
)

In [ ]:
px.histogram(x=temp.groupby(index.names[0]).size(), nbins=50).update_layout(  # type: ignore
    margin=config.MARGIN,  # type: ignore
    title={
        "text": f"Store {value} lengths distribution",
        "x": 0.5,
    },
    xaxis_title="length",
    yaxis_title="amount",
)

### Exogenous data transformation

In [ ]:
# Expanding exogenous data with additional features based on the date index
ft = ExogenousDataTransformer()
X_temp_train = ft.fit_transform(X_temp_train)
X_temp_test = ft.transform(X_temp_test)
X_temp_train.head()  # type: ignore

### Clustering

In [ ]:
# Aligning exogenous variables with target by index
X_temp_train = temp[[]].join(X_temp_train)

In [ ]:
# Clustering series with similar shape, statistics and magnitude to enhance the
# global forecasting quality per cluster
clusterer = SmoothErraticClusterer(w=90)

dir_path = Path.cwd().parent / "models"
path = dir_path / (
    "_".join([value, demand, clusterer.get_tag("object_type")])  # type: ignore
    + ".joblib"
)

if not Path.is_file(path):
    clusterer.fit(X=temp)
    dump(clusterer, path)
else:
    clusterer = load(path)

clusterer.metrics

In [ ]:
# Assigning cluster labels to instances as a primary index
labels = pd.Series(clusterer.labels, name="cluster").rename_axis("index")
temp = (
    temp.join(labels)
    .set_index("cluster", append=True)
    .reorder_levels(["cluster"] + index.names)  # type: ignore
    .sort_index()
)
X_temp_train = temp[[]].join(X_temp_train)
X_temp_test = (
    X_temp_test.join(labels)  # type: ignore
    .set_index("cluster", append=True)
    .reorder_levels(["cluster"] + index.names)  # type: ignore
    .sort_index()
)
labels.value_counts()

In [ ]:
# Visualizing clusters in order to evaluate homogeneity of their instances
clusteredSeriesPlot(data=temp, title=f"Ecuador clustered stores {value}")

In [ ]:
temp.groupby("cluster").size().sort_values(ascending=False)

### Modeling

In [ ]:
# Fitting forecasting model per cluster based on its size and with all exogenous
# data accommodation
forecaster = CommonForecaster(
    dir_path=dir_path, value=value, demand=demand, n_jobs=cpu_count()
)
forecaster.fit(y=temp, X=X_temp_train, fh=fh)

In [ ]:
# Forecasting with the same exogenous data to complete target variable and stack
# to other exogenous features for further modeling
forecasts = forecaster.predict(X=X_temp_test)
result = pd.concat(
    [
        mapper.transform(transactions)
        .interpolate(method="linear")  # type: ignore
        .bfill()
        .ffill()
        .assign(group="train"),
        forecasts.droplevel(0).assign(group="forecast"),  # type: ignore
    ]
)
result = mapper.inverse_transform(result)
result.head()

In [ ]:
forecastedSeriesPlot(
    data=result,  # type: ignore
    n_series=4,
    title=f"Ecuador forecasted stores {value}",
    seed=134,
)

In [ ]:
transactions = result.drop(columns="group")  # type: ignore

# Sales

In [ ]:
# Proposed solution disclosing CommonPipeline stepwise strategy
X_exog_train = X_exog_train.join(oil).join(stores).join(transactions)
X_exog_test = X_exog_test.join(oil).join(stores).join(transactions)
value = "sales"
sales.head()

## Multiindex reduction

In [ ]:
# Replacing multiindex with a single index to avoid further pandas index manipulation
# issues
mapper = Mapper()
sales = mapper.fit_transform(sales)
X_exog_train = mapper.transform(X_exog_train)
X_exog_test = mapper.transform(X_exog_test)

## Classification

In [ ]:
# Series behavior classification to define an appropriate processing and modeling
result = DemandClassifier().fit_transform(sales)
result["class"].value_counts()  # type: ignore

In [ ]:
px.scatter(data_frame=result, x="adi", y="cv2", color="class").update_layout(
    title={
        "text": f"{value.capitalize()} demand classification",
        "x": 0.5,
    },
    margin=config.MARGIN,  # type: ignore
)

In [ ]:
sales = (
    sales.join(result["class"])  # type: ignore
    .set_index("class", append=True)
    .reorder_levels(["class"] + sales.index.names)  # type: ignore
    .sort_index()
)

sales.head()

In [ ]:
# Visualizing examples of each demand class in train data
classifiedSeriesPlot(data=sales, title=f"Variations if {value} demand classes", seed=13)

## Workflow per demand class

### Smooth & erratic

In [ ]:
# Only smooth demand stepwise solution presented, the erratic is passing the same
# processing strategies and core fitting models
demand = "smooth"
temp = sales.loc[demand]

#### Plateau detection

In [ ]:
# Defining a temporal inactivity expressed as a plateau - zero values sequence with
# a specific length and leave only data which follows the last plateau if any defined
pld = PlateauDetector(w=2 * config.SEASONAL_PERIOD, value=0, truncate=True)  # type: ignore
temp = pld.fit_transform(temp)

#### Change point detection

In [ ]:
# Defining an inflection point with specific requirements where the most significant
# trend behavior change was occurred
seriesPlot(
    data=temp,  # type: ignore
    demand=demand,
    n_series=4,
    title=f"Ecuador store {value} with change point detection",
    w=config.MIN_LENGTH,  # type: ignore
    cpd=True,
    seed=666,
)

In [ ]:
# Leave only data which follows right after this point if any exists
cpd = ChangePointDetector(w=config.MIN_LENGTH, truncate=True)  # type: ignore
temp = cpd.fit_transform(temp)

#### Outliers detection

In [ ]:
# Defining outliers as a significant temporal demand with holidays and promotions
# data taking into account
seriesPlot(
    data=temp,  # type: ignore
    demand=demand,
    X_exog=X_exog_train,
    exog_column="on_promotion",
    n_series=4,
    title=f"Ecuador store {value} with outliers detection",
    w=2 * config.SEASONAL_PERIOD,  # type: ignore
    od=True,
    seed=69,
)

In [ ]:
# Remove selected points that are not related to holidays and promotion days
od = OutlierDetector(demand=demand, exog_column="on_promotion", truncate=True)
temp = od.fit_transform(temp.join(X_exog_train))  # type: ignore

#### Missing values imputation

In [ ]:
# Interpolating missing values if any present
index = temp.index  # type: ignore
temp = temp.groupby(index.names[0]).transform(  # type: ignore
    lambda x: x.interpolate(method="linear").bfill().ffill()
)

In [ ]:
px.histogram(x=temp.groupby(index.names[0]).size(), nbins=50).update_layout(  # type: ignore
    margin=config.MARGIN,  # type: ignore
    title={
        "text": f"Store {value} lengths distribution",
        "x": 0.5,
    },
    xaxis_title="length",
    yaxis_title="amount",
)

#### Exogenous data transformation

In [ ]:
# Expanding exogenous data with additional features based on the date index
ft = ExogenousDataTransformer()
X_temp_train = ft.fit_transform(X_exog_train)
X_temp_test = ft.transform(X_exog_test)
X_temp_train.head()  # type: ignore

#### Clustering

In [ ]:
# Aligning exogenous variables with target by index
X_temp_train = temp[[]].join(X_temp_train)
X_temp_test = X_temp_test.loc[temp.droplevel(-1).index.unique()]  # type: ignore

In [ ]:
# Clustering series with similar shape, statistics and magnitude to enhance the
# global forecasting quality per cluster
clusterer = SmoothErraticClusterer(w=90)

dir_path = Path.cwd().parent / "models"
path = dir_path / (
    "_".join([value, demand, clusterer.get_tag("object_type")])  # type: ignore
    + ".joblib"
)

if not Path.is_file(path):
    clusterer.fit(X=temp)
    dump(clusterer, path)
else:
    clusterer = load(path)

clusterer.metrics

In [ ]:
# Assigning cluster labels to instances as a primary index
labels = pd.Series(clusterer.labels, name="cluster").rename_axis("index")
temp = (
    temp.join(labels)
    .set_index("cluster", append=True)
    .reorder_levels(["cluster"] + index.names)  # type: ignore
    .sort_index()
)
X_temp_train = temp[[]].join(X_temp_train)
X_temp_test = (
    X_temp_test.join(labels)  # type: ignore
    .set_index("cluster", append=True)
    .reorder_levels(["cluster"] + index.names)  # type: ignore
    .sort_index()
)
labels.value_counts().head()

In [ ]:
# Visualizing clusters in order to evaluate homogeneity of their instances
# The loc was used to only control the output plot size
clusteredSeriesPlot(
    data=temp.loc[[*range(4)]], title=f"Ecuador clustered stores {value}"
)

In [ ]:
temp.groupby("cluster").size().sort_values(ascending=False).head()

#### Modeling

In [ ]:
# Fitting forecasting model per cluster based on its size and with all exogenous
# data accommodation
forecaster = CommonForecaster(
    dir_path=dir_path, value=value, demand=demand, n_jobs=cpu_count()
)
forecaster.fit(y=temp, X=X_temp_train, fh=fh)

In [ ]:
forecasts = forecaster.predict(X=X_temp_test)
result = mapper.inverse_transform(
    pd.concat(
        [
            temp.assign(group="train"),
            forecasts.assign(group="forecast"),  # type: ignore
        ]
    ).droplevel(0)
)
result.head()

In [ ]:
forecastedSeriesPlot(
    data=result,  # type: ignore
    n_series=4,
    title=f"Ecuador forecasted stores {value}",
    seed=69,
)

### Intermittent & lumpy

In [ ]:
# Only intermittent demand stepwise solution presented, the lumpy is passing the same
# processing strategies and core fitting models
demand = "intermittent"
temp = sales.loc[demand]

#### Outliers Detection

In [ ]:
# Defining outliers as a significant temporal demand with holidays and promotions
# data taking into account
seriesPlot(
    data=temp,  # type: ignore
    demand=demand,
    X_exog=X_exog_train,
    exog_column="on_promotion",
    n_series=4,
    title=f"Ecuador store {value} with outliers detection",
    w=2 * config.SEASONAL_PERIOD,  # type: ignore
    od=True,
    seed=69,
)

In [ ]:
# Remove selected points that are not related to holidays and promotion days
od = OutlierDetector(demand=demand, exog_column="on_promotion", truncate=True)
temp = od.fit_transform(temp.join(X_exog_train))  # type: ignore

#### Missing values imputation

In [ ]:
# Interpolating missing values if any present
index = temp.index  # type: ignore
temp = temp.groupby(temp.index.names[0]).transform(  # type: ignore
    lambda x: x.fillna(
        x.rolling(window=config.SEASONAL_PERIOD, min_periods=1).median()  # type: ignore
    ).fillna(0)
)

In [ ]:
px.histogram(x=temp.groupby(index.names[0]).size(), nbins=50).update_layout(  # type: ignore
    margin=config.MARGIN,  # type: ignore
    title={
        "text": f"Store {value} lengths distribution",
        "x": 0.5,
    },
    xaxis_title="length",
    yaxis_title="amount",
)

#### Exogenous features transformation

In [ ]:
# Expanding exogenous data with additional features based on the date index
ft = ExogenousDataTransformer()
X_temp_train = ft.fit_transform(X_exog_train)
X_temp_test = ft.transform(X_exog_test)
X_temp_train.head()  # type: ignore

#### Clustering

In [ ]:
# Aligning exogenous variables with target by index
X_temp_train = temp[[]].join(X_temp_train)
X_temp_test = X_temp_test.loc[temp.droplevel(-1).index.unique()]  # type: ignore

In [ ]:
# Clustering series with similar shape, statistics and magnitude to enhance the
# global forecasting quality per cluster
clusterer = IntermittentLumpyClusterer()

dir_path = Path.cwd().parent / "models"
path = dir_path / (
    "_".join([value, demand, clusterer.get_tag("object_type")])  # type: ignore
    + ".joblib"
)

if not Path.is_file(path):
    clusterer.fit(X=temp)
    dump(clusterer, path)
else:
    clusterer = load(path)

clusterer.metrics

In [ ]:
# Assigning cluster labels to instances as a primary index
labels = pd.Series(clusterer.labels, name="cluster").rename_axis("index")
temp = (
    temp.join(labels)
    .set_index("cluster", append=True)
    .reorder_levels(["cluster"] + index.names)  # type: ignore
    .sort_index()
)
X_temp_train = temp[[]].join(X_temp_train)
X_temp_test = (
    X_temp_test.join(labels)  # type: ignore
    .set_index("cluster", append=True)
    .reorder_levels(["cluster"] + index.names)  # type: ignore
    .sort_index()
)
labels.value_counts()

In [ ]:
# Visualizing clusters in order to evaluate homogeneity of their instances
clusteredSeriesPlot(data=temp, title=f"Ecuador clustered stores {value}")

In [ ]:
temp.groupby(level=0).size().sort_values(ascending=False)

#### Modeling

In [ ]:
# Fitting forecasting model per cluster based on its size and with all exogenous
# data accommodation

forecaster = CommonForecaster(
    dir_path=dir_path, value=value, demand=demand, n_jobs=cpu_count()
)
forecaster.fit(y=temp, X=X_temp_train, fh=fh)

In [ ]:
forecasts = forecaster.predict(X=X_temp_test)
result = mapper.inverse_transform(
    pd.concat(
        [
            temp.assign(group="train"),
            forecasts.assign(group="forecast"),  # type: ignore
        ]
    ).droplevel(0)
)
result.head()

In [ ]:
forecastedSeriesPlot(
    data=result,  # type: ignore
    n_series=4,
    title=f"Ecuador forecasted stores {value}",
    seed=666,
)

## End-to-end forecasting pipeline

In [ ]:
# To fit models for other demand classes the main e2e-model with all described above
# steps in a single fit/predict pipeline is used

# To apply it some inverse transformations are required returning to original data
sales = mapper.inverse_transform(X=sales.droplevel(0))
X_exog_train = mapper.inverse_transform(X=X_exog_train)
X_exog_test = mapper.inverse_transform(X=X_exog_test)

In [ ]:
forecaster = E2EForecaster(dir_path=Path.cwd().parent / "models")
forecaster.fit(y=sales, X=X_exog_train, fh=config.FH_SIZE)  # type: ignore

In [ ]:
forecasts = forecaster.predict(X=X_exog_test)
result = pd.concat(
    [
        sales.assign(group="train"),  # type: ignore
        forecasts.assign(group="forecast"),  # type: ignore
    ]
)
result.head()

In [ ]:
forecastedSeriesPlot(
    data=result, n_series=10, title=f"Ecuador forecasted stores {value}", seed=666
)  # type: ignore

In [ ]:
# Post fit core models quality analysis based on train sample size and demand class
table = pd.DataFrame()
for demand in forecaster.models:
    for group in forecaster.models[demand].forecaster.models:
        t = load(forecaster.models[demand].forecaster.models[group])
        del t["model"]
        t["demand"] = demand
        t["group"] = group
        table = pd.concat([table, pd.Series(t).to_frame().T])

px.scatter(data_frame=table, x="y_size", y="score", color="demand").update_layout(
    title={
        "text": f"""{table.shape[0]} models for {result.droplevel(-1).index.nunique()} 
        {value} series""",
        "x": 0.5,
    },
    xaxis_title="train samples quantity",
    yaxis_title="RMSLE",
    margin=config.MARGIN,  # type: ignore
)

# Submission

In [ ]:
# Updating submission with died products zeros filling
test = pd.read_csv(
    filepath_or_buffer="../data/raw/test.csv",
    parse_dates=["date"],
    index_col=["id", "store_nbr", "family", "date"],
    engine="pyarrow",
)[[]]
submission = pd.read_csv(
    filepath_or_buffer="../data/raw/sample_submission.csv",
    index_col="id",
    engine="pyarrow",
)

submission = (
    submission[[]].join(test.join(result[value]).fillna(0)).reset_index()[["id", value]]
)
submission.head()

In [ ]:
submission.to_csv("../data/raw/sample_submission.csv", index=False)